In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

In [3]:
# Read in ridership data 
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026'

In [4]:
ridership_data = pd.read_csv(f"{GCS_FILE_PATH}/ridership_all.csv")

/tmp/ipykernel_6690/831429262.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  ridership_data = pd.read_csv(f"{GCS_FILE_PATH}/ridership_all.csv")


In [5]:
ridership_data = ridership_data.sort_values(by = 'stop_name')
ridership_data.head(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
16252,Gold Coast Transit,16,NaN,VNACLR1,.,34.343645,-119.294028,Weekday,0.000000,3.000000,2025-05-01,2025-05-31
56244,Samtrans,ECR,NaN,345017,1000 El Camino Real-Menlo College,37.457839,-122.190513,Saturday,9.400000,16.200000,2025-08-01,2025-08-31
56245,Samtrans,ECR,NaN,345017,1000 El Camino Real-Menlo College,37.457839,-122.190513,Sunday,8.800000,15.600000,2025-08-01,2025-08-31
56246,Samtrans,ECR,NaN,345017,1000 El Camino Real-Menlo College,37.457839,-122.190513,Weekday,9.523810,24.571429,2025-08-01,2025-08-31
17509,Golden Gate Transit,130,NaN,40469,1011 Andersen Dr,NaN,NaN,Weekday,1.909091,0.000000,2025-09-01,2025-09-30


In [6]:
ridership_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 71703 entries, 16252 to 45845
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   organization_name         71703 non-null  object 
 1   route_id                  65173 non-null  object 
 2   route_name                27000 non-null  object 
 3   stop_id                   71293 non-null  object 
 4   stop_name                 56894 non-null  object 
 5   stop_lat                  21395 non-null  float64
 6   stop_lon                  21395 non-null  float64
 7   day_type                  71703 non-null  object 
 8   average_daily_boardings   71703 non-null  float64
 9   average_daily_alightings  63461 non-null  float64
 10  start_date                71703 non-null  object 
 11  end_date                  71703 non-null  object 
dtypes: float64(4), object(8)
memory usage: 7.1+ MB


In [7]:
ridership_data['route_id'] = ridership_data['route_id'].apply(
    lambda x: str(int(x)) if isinstance(x, float) and not pd.isna(x) else str(x)
)

In [8]:
ridership_data_weekday = ridership_data[ridership_data['day_type'] == "Weekday"]

In [9]:
stops_aggregated_weekday = pd.read_csv(f"{GCS_FILE_PATH}/stops_aggregated_weekday.csv")

In [10]:
agency_mapping = {
    'Gold Coast Schedule': 'Gold Coast Transit',
    'SamTrans Schedule': 'Samtrans',
    'Big Blue Bus Schedule': 'Big Blue Bus',
    'Foothill Schedule': 'Foothill Transit',
    'San Diego Schedule': 'SDMTS',
    'Golden Gate Park Shuttle Schedule': 'Golden Gate Park Shuttle',
    'Fresno Schedule': 'Fresno County',
    'OmniTrans Schedule': 'Omnitrans',
    'Sacramento Schedule': 'SacRT Bus',
    'Bay Area 511 BART Schedule': 'San Francisco Bay Area Rapid Transit District',
    'Riverside Schedule': 'Riverside Transit',
    'OCTA Schedule': 'Orange County Transportation Authority',
    'Santa Cruz Schedule': 'Santa Cruz Metro',
    'Long Beach Schedule': 'Long Beach Transit',
    'Caltrain Schedule': 'Caltrain',
    'SBMTD Schedule': 'SBMTD',
    'Culver City Schedule': 'Culver City Bus',
    'Bay Area 511 Golden Gate Transit Schedule': 'Golden Gate Transit'
}

In [11]:
# Create organization_name based on the agency mapping
stops_aggregated_weekday['organization_name'] = stops_aggregated_weekday['name'].map(agency_mapping)

In [12]:
# Upon Checking found that Samtrans Transit Route Id has -216 in the stops aggregated data, while ridership data we have doesnot. So removing -216 from stopsa_aggregated data 
stops_aggregated_weekday.loc[
    stops_aggregated_weekday['name'] == "SamTrans Schedule",
    'route_id'
] = stops_aggregated_weekday.loc[
    stops_aggregated_weekday['name'] == "SamTrans Schedule",
    'route_id'
].str.replace('-216$', '', regex=True)

In [13]:
# Upon checking, found that Gold Coast Schedule stop_ids have some issues:
# 1. Some stop_ids contain ':' which prevents correct merge with ridership data
# 2. Two specific stop_ids ('GCTDMAIN1' and 'SAClMAI1') do not match the ridership data stop_ids ('GCTDMAIN' and 'SACLMAI1')
# So we remove colons and update these two stop_ids before merging
stops_aggregated_weekday.loc[
    stops_aggregated_weekday['name'] == "Gold Coast Schedule",
    'stop_id'
] = stops_aggregated_weekday.loc[
    stops_aggregated_weekday['name'] == "Gold Coast Schedule",
    'stop_id'
].str.replace(':', '', regex=False).replace({
    'GCTDMAIN1': 'GCTDMAIN',
    'SAClMAI1': 'SACLMAI1'
})

In [14]:
# Upon checking, found that Some stops are unmatched because Big Blue Bus ridership uses negative stop_codes.
# These correspond to valid stop_codes in the stop data (same route and stop_name).
# The mapping below replaces negative ridership stop_codes with the correct stop_codes
# so they can match during the merge.

# Apply only for Big Blue Bus
mask_bb = ridership_data_weekday['organization_name'] == 'Big Blue Bus'

# Convert to int first
ridership_data_weekday.loc[mask_bb, 'stop_id'] = (
    ridership_data_weekday.loc[mask_bb, 'stop_id']
    .astype(int)
    .replace({
        -4: 'MNSWSMNF',
        -5: '014MCHNN',
        -6: '014PCONF',
        -7: '014PCOSF',
        -8: '017PRLNN',
        -9: '017PRLSF',
        -13: 'PRL014EF',
        -14: 'PRL014WN',
        -18: 'SMCBUNPL'
    })
    .astype(str)
)


stops_aggregated_weekday.loc[
    stops_aggregated_weekday['organization_name'] == 'Big Blue Bus',
    'stop_code'
] = stops_aggregated_weekday.loc[
    stops_aggregated_weekday['organization_name'] == 'Big Blue Bus',
    'stop_code'
].astype(str).str.strip()

In [15]:
# Upon checking found that some route_ids are differently named in ridership data compared to stop_data so changing that to match
# Filter for Culver City Bus
mask_cc = ridership_data_weekday['organization_name'] == 'Culver City Bus'

# Standardize route_id in ridership
ridership_data_weekday.loc[mask_cc, 'route_id'] = (
    ridership_data_weekday.loc[mask_cc, 'route_id']
    .replace({
        '1C1': '1c1',
        'Rapid 6': '6R'
    })
)

# Standardize route_id in stops
stops_aggregated_weekday.loc[
    stops_aggregated_weekday['organization_name'] == 'Culver City Bus',
    'route_id'
] = stops_aggregated_weekday.loc[
    stops_aggregated_weekday['organization_name'] == 'Culver City Bus',
    'route_id'
].replace({
    '1C1': '1c1',
    'Rapid 6': '6R'
})

In [16]:
# Upon checking found that Long Beach transit stop_id has leading zeros like 0110, 0220 and so on
stops_aggregated_weekday.loc[
    stops_aggregated_weekday['organization_name'] == 'Long Beach Transit',
    'stop_id'
] = stops_aggregated_weekday.loc[
    stops_aggregated_weekday['organization_name'] == 'Long Beach Transit',
    'stop_id'
].str.replace(r'^0+', '', regex=True)

In [17]:
# Upon Checking found that Caltrain has 

In [18]:
stops_aggregated_weekday.head(5)

,feed_key,stop_id,stop_name,route_id,route_type,n_trips,n_routes,gtfs_dataset_key,name,route_long_name,stop_code,pt_geom,organization_name
0,1eb334a547f5c2cdbdce3f7cc7b03e2b,1000,SLP DR & 43RD AVE (SB),106,3.0,6,1,98588c1726b50acbef430783ab0e734e,Sacramento Schedule,LAND PARK COMMUTER,1000,NaN,SacRT Bus
1,1eb334a547f5c2cdbdce3f7cc7b03e2b,1000,SLP DR & 43RD AVE (SB),227,3.0,1,1,98588c1726b50acbef430783ab0e734e,Sacramento Schedule,SOUTH LAND PARK - GREENHAVEN DR,1000,NaN,SacRT Bus
2,1eb334a547f5c2cdbdce3f7cc7b03e2b,1001,SLP DR & 43RD AVE (SB),106,3.0,6,1,98588c1726b50acbef430783ab0e734e,Sacramento Schedule,LAND PARK COMMUTER,1001,NaN,SacRT Bus
3,1eb334a547f5c2cdbdce3f7cc7b03e2b,1001,SLP DR & 43RD AVE (SB),227,3.0,1,1,98588c1726b50acbef430783ab0e734e,Sacramento Schedule,SOUTH LAND PARK - GREENHAVEN DR,1001,NaN,SacRT Bus
4,1eb334a547f5c2cdbdce3f7cc7b03e2b,1002,SLP DR & 47TH AVE (SB),106,3.0,6,1,98588c1726b50acbef430783ab0e734e,Sacramento Schedule,LAND PARK COMMUTER,1002,NaN,SacRT Bus


In [19]:
agencies_with_route = [
    'Foothill Transit',
    'Gold Coast Transit',
    'Golden Gate Transit',
    'Long Beach Transit',
    'Orange County Transportation Authority'
    'Omnitrans',    
    'SacRT Bus'
    'Samtrans',
    'SDMTS'
]

agencies_with_route_stop_code_match = [
    'Culver City Bus',
    'Big Blue Bus',
    'Riverside Transit'
]


agencies_without_route = ['San Francisco Bay Area Rapid Transit District', 
                          'Caltrain', 
                          'Fresno County',
                          'SBMTD'
                         ]


agencies_without_route_stop_code_match = [
                          'Santa Cruz Metro',
                          'Golden Gate Park Shuttle',
                         ]

In [20]:
# Split the ridership data 
ridership_with_route = ridership_data_weekday[
    ridership_data_weekday['organization_name'].isin(agencies_with_route)
]

ridership_without_route = ridership_data_weekday[
    ridership_data_weekday['organization_name'].isin(agencies_without_route)
]

ridership_with_route_stopcode_match = ridership_data_weekday[
    ridership_data_weekday['organization_name'].isin(agencies_with_route_stop_code_match)
]

ridership_without_route_stopcode_match = ridership_data_weekday[
    ridership_data_weekday['organization_name'].isin(agencies_without_route_stop_code_match)
]


In [21]:
ridership_without_route_stopcode_match.organization_name.unique()

array(['Golden Gate Park Shuttle'], dtype=object)

In [22]:
# Merge with route_id - removing Big Blue Bus, since it provides stop_code 
merge_with_route = pd.merge(
    ridership_with_route,
    stops_aggregated_weekday,
    on=['organization_name', 'route_id', 'stop_id'],
    how='left',
    indicator=True,
    suffixes=('_x','_y')
)

merge_counts = merge_with_route['_merge'].value_counts()
print(merge_counts)

_merge
both          12532
left_only       997
right_only        0
Name: count, dtype: int64


In [23]:
# Merge without route_id
merge_without_route = pd.merge(
    ridership_without_route,
    stops_aggregated_weekday,
    on=['organization_name', 'stop_id'],
    how='left',
    indicator=True,
    suffixes=('_x','_y')
)

merge_without_route['_merge'].value_counts()

_merge
both          1622
left_only       77
right_only       0
Name: count, dtype: int64

In [26]:
merge_with_route_stopcode = pd.merge(
    ridership_with_route_stopcode_match,
    stops_aggregated_weekday,
    left_on=['organization_name','route_id','stop_id'],  # ridership stop_id
    right_on=['organization_name','route_id','stop_code'], # cleaned stop_code
    how='left',
    indicator=True
)

merge_with_route_stopcode['_merge'].value_counts()

_merge
both          4073
left_only      785
right_only       0
Name: count, dtype: int64

In [27]:
merge_without_route_stopcode = pd.merge(
    ridership_without_route_stopcode_match,
    stops_aggregated_weekday,
    left_on=['organization_name', 'stop_id'],  # ridership stop_id
    right_on=['organization_name', 'stop_code'], # cleaned stop_code
    how='left',
    indicator=True
)

merge_without_route_stopcode ['_merge'].value_counts()

_merge
both          18
left_only      0
right_only     0
Name: count, dtype: int64

In [28]:
# Count how many rows matched (both) vs unmatched (left_only) per organization
match_summary = merge_with_route_stopcode.groupby(['organization_name', '_merge']).size().unstack(fill_value=0)
print(match_summary)

_merge             left_only  right_only  both
organization_name                             
Big Blue Bus             250           0  1065
Culver City Bus           23           0   453
Riverside Transit        512           0  2555


/tmp/ipykernel_6690/1411928191.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  match_summary = merge_with_route_stopcode.groupby(['organization_name', '_merge']).size().unstack(fill_value=0)


In [29]:
# Count how many rows matched (both) vs unmatched (left_only) per organization
match_summary1 = merge_with_route.groupby(['organization_name', '_merge']).size().unstack(fill_value=0)
print(match_summary1)

_merge               left_only  right_only  both
organization_name                               
Foothill Transit             7           0  2423
Gold Coast Transit           0           0  1032
Golden Gate Transit        172           0   661
Long Beach Transit          86           0  3245
SDMTS                      732           0  5171


/tmp/ipykernel_6690/782609038.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  match_summary1 = merge_with_route.groupby(['organization_name', '_merge']).size().unstack(fill_value=0)


In [30]:
# Count how many rows matched (both) vs unmatched (left_only) per organization
match_summary2 = merge_without_route.groupby(['organization_name', '_merge']).size().unstack(fill_value=0)
print(match_summary2)

_merge                                         left_only  right_only  both
organization_name                                                         
Caltrain                                              30           0     0
Fresno County                                         47           0  1571
San Francisco Bay Area Rapid Transit District          0           0    51


/tmp/ipykernel_6690/2872024187.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  match_summary2 = merge_without_route.groupby(['organization_name', '_merge']).size().unstack(fill_value=0)


In [29]:
# Count how many rows matched (both) vs unmatched (left_only) per organization
match_summary3 = merge_without_route_stopcode.groupby(['organization_name', '_merge']).size().unstack(fill_value=0)
print(match_summary3)

_merge                    left_only  right_only  both
organization_name                                    
Golden Gate Park Shuttle          0           0    18


/tmp/ipykernel_5303/1814182095.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  match_summary3 = merge_without_route_stopcode.groupby(['organization_name', '_merge']).size().unstack(fill_value=0)


In [24]:
merge_without_route_golden = merge_without_route[merge_without_route['organization_name'] == "Golden Gate Park Shuttle"]

In [25]:
merge_without_route_golden.head(5)

,organization_name,route_id_x,route_name,stop_id,stop_name_x,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,feed_key,stop_name_y,route_id_y,route_type,n_trips,n_routes,gtfs_dataset_key,name,route_long_name,stop_code,pt_geom,_merge
0,Golden Gate Park Shuttle,nan,NaN,7607,10th Ave/ De Young EB,NaN,NaN,Weekday,5.862069,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,Golden Gate Park Shuttle,nan,NaN,7608,10th Ave/ De Young WB,NaN,NaN,Weekday,6.800766,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
46,Golden Gate Park Shuttle,nan,NaN,7611,8th Ave EB,NaN,NaN,Weekday,8.674330,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
47,Golden Gate Park Shuttle,nan,NaN,7610,8th Ave WB,NaN,NaN,Weekday,7.551724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
55,Golden Gate Park Shuttle,nan,NaN,7609,Academy of Sciences,NaN,NaN,Weekday,12.881226,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
